# Collaborative Filtering

### Metody
- User-based filtering
    - Obliczenie macierzy podobieństwa użytkowników
    - Znalezienie k najbliższych sąsiadów dla użytkownika
    - Przewidywanie ocen na podstawie ocen sąsiadów
    - Wybranie k najlepszych gier
- Item-based filtering
    - Obliczenie macierzy podobieństwa gier
    - Znalezienie gier podobnych do ocenionych przez użytkownika
    - Przewidywanie ocen na podstawie podobieństwa gier
    - Wybranie k najlepszych gier
- SVD (Matrix factorization)
    - Inicjalizacja macierzy cech użytkowników i gier
    - Trenowanie modelu przez minimalizację błędu
    - Przewidywanie ocen dla nieocenionych gier
    - Wybranie k najlepszych gier

User-based - „Użytkownicy podobni do Bartka lubią te gry → poleć je Bartkowi.”

Item-based - „Bartek oglądał Breaking Bad. Ludzie, którzy oglądają Breaking Bad, często oglądają Better Call Saul.”

### Metryki
- Cosinusowe podobieństwo
- Odległość euklidesowa???
- Korelacja Pearsona

In [21]:
import pandas as pd

games = pd.read_json("games-vault/chatbot/data/steam_app_details.jsonl", lines=True, nrows=1000)
users = pd.read_json("games-vault/chatbot/data/steam_users_ratings.json")

### Przygotowanie danych i stworzenie macierzy user/game

In [22]:
players_data = []
for player in users['players']:
    for review in player['reviews']:
        players_data.append({
            'playerId': player['playerId'],
            'gameId': review['gameId'],
            'rating': review['rating']
        })

df = pd.DataFrame(players_data)

In [23]:
df

,playerId,gameId,rating
0,0,4,7
1,0,13,4
2,0,28,10
3,0,37,3
4,0,45,8
...,...,...,...
2249,499,5,9
2250,499,12,3
2251,499,21,7
2252,499,33,10


In [24]:
user_item_matrix = df.pivot_table(
    index='playerId', 
    columns='gameId', 
    values='rating'
).fillna(0)

In [25]:
user_item_matrix

gameId,0,1,2,3,4,5,6,7,8,9,...,41,42,43,44,45,46,47,48,49,50
playerId,,,,,,,,,,,,,,,,,,,,,
0,0.0,0.0,0.0,0.0,7.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,8.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,9.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,6.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,7.0,0.0,0.0,0.0,0.0,0.0,5.0,0.0,0.0
3,10.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,8.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,6.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
495,0.0,0.0,7.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,3.0,0.0,0.0,0.0,0.0,0.0
496,0.0,0.0,0.0,0.0,0.0,0.0,0.0,6.0,0.0,0.0,...,0.0,4.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
497,5.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.0,0.0,0.0


In [26]:
print(f"Macierz ocen: {user_item_matrix.shape}")
print(f"Liczba ocen: {len(df)}")
print(f"Gęstość danych: {len(df) / (user_item_matrix.shape[0] * user_item_matrix.shape[1]) * 100:.2f}%")

Macierz ocen: (500, 51)
Liczba ocen: 2254
Gęstość danych: 8.84%


### Funkcje rekomendacji

In [27]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import StandardScaler
from scipy.spatial.distance import pdist, squareform
from scipy.stats import pearsonr
import warnings
warnings.filterwarnings('ignore')

#### USER-BASED

In [28]:
def user_based_cf(user_id, n_recommendations=5, similarity='cosine'):
    """
    User-Based Collaborative Filtering
    Bez użycia zewnętrznych bibliotek
    """
    # Pobierz oceny użytkownika
    user_idx = user_item_matrix.index.get_loc(user_id)
    user_ratings = user_item_matrix.iloc[user_idx].values
    
    # Znajdź gry nieocenione przez użytkownika
    unrated_mask = user_ratings == 0
    unrated_indices = np.where(unrated_mask)[0]
    
    if len(unrated_indices) == 0:
        return "Użytkownik ocenił już wszystkie gry"
    
    # Oblicz podobieństwo między użytkownikami
    if similarity == 'cosine':
        # Cosinusowe podobieństwo
        similarity_matrix = cosine_similarity(user_item_matrix)
    elif similarity == 'pearson':
        # Korelacja Pearsona
        similarity_matrix = np.corrcoef(user_item_matrix)
        similarity_matrix = np.nan_to_num(similarity_matrix, nan=0)
    elif similarity == 'euclidean':
        # Odległość euklidesowa
        dist_matrix = squareform(pdist(user_item_matrix, metric='euclidean'))
        similarity_matrix = 1 / (1 + dist_matrix)
    else:
        similarity_matrix = cosine_similarity(user_item_matrix)
    
    # Podobieństwa do innych użytkowników
    user_similarities = similarity_matrix[user_idx]
    
    # Przewiduj oceny dla nieocenionych gier
    predictions = {}
    
    for game_idx in unrated_indices:
        game_ratings = user_item_matrix.iloc[:, game_idx].values
        
        # Tylko użytkownicy, którzy ocenili tę grę
        rated_users = game_ratings > 0
        if not rated_users.any():
            continue
        
        # Ważona średnia
        weighted_sum = np.sum(user_similarities[rated_users] * game_ratings[rated_users])
        similarity_sum = np.sum(user_similarities[rated_users])
        
        if similarity_sum > 0:
            game_id = user_item_matrix.columns[game_idx]
            predictions[game_id] = weighted_sum / similarity_sum
    
    # Sortuj i zwróć rekomendacje
    recommendations = sorted(predictions.items(), key=lambda x: x[1], reverse=True)[:n_recommendations]
    return recommendations

#### ITEM-BASED

In [29]:
def item_based_cf(user_id, n_recommendations=5, similarity='cosine'):
    """
    Item-Based Collaborative Filtering
    Bez użycia zewnętrznych bibliotek
    """
    # Pobierz oceny użytkownika
    user_idx = user_item_matrix.index.get_loc(user_id)
    user_ratings = user_item_matrix.iloc[user_idx].values
    
    # Znajdź gry, które użytkownik ocenił
    rated_mask = user_ratings > 0
    rated_indices = np.where(rated_mask)[0]
    
    if len(rated_indices) == 0:
        return "Użytkownik nie ocenił żadnej gry"
    
    # Oblicz podobieństwo między grami
    if similarity == 'cosine':
        item_similarity = cosine_similarity(user_item_matrix.T)
    elif similarity == 'pearson':
        item_similarity = np.corrcoef(user_item_matrix.T)
        item_similarity = np.nan_to_num(item_similarity, nan=0)
    elif similarity == 'euclidean':
        dist_matrix = squareform(pdist(user_item_matrix.T, metric='euclidean'))
        item_similarity = 1 / (1 + dist_matrix)
    else:
        item_similarity = cosine_similarity(user_item_matrix.T)
    
    # Przewiduj oceny dla nieocenionych gier
    predictions = {}
    
    for game_idx in range(len(user_item_matrix.columns)):
        if game_idx in rated_indices:
            continue
        
        # Znajdź podobieństwa do ocenionych gier
        similarities = item_similarity[game_idx][rated_indices]
        rated_ratings = user_ratings[rated_indices]
        
        # Ważona średnia
        weighted_sum = np.sum(similarities * rated_ratings)
        similarity_sum = np.sum(similarities)
        
        if similarity_sum > 0:
            game_id = user_item_matrix.columns[game_idx]
            predictions[game_id] = weighted_sum / similarity_sum
    
    # Sortuj i zwróć rekomendacje
    recommendations = sorted(predictions.items(), key=lambda x: x[1], reverse=True)[:n_recommendations]
    return recommendations

#### SVD - Matrix Factorization

In [30]:
class CustomSVD:
    """
    Własna implementacja SVD dla collaborative filtering
    Używa gradient descent do znalezienia macierzy cech
    """
    def __init__(self, n_factors=20, learning_rate=0.005, regularization=0.02, n_epochs=30):
        self.n_factors = n_factors
        self.learning_rate = learning_rate
        self.regularization = regularization
        self.n_epochs = n_epochs
        self.user_factors = None
        self.item_factors = None
        self.user_bias = None
        self.item_bias = None
        self.global_mean = None
        
    def fit(self, ratings_df):
        """
        Trenuj model na danych
        ratings_df: DataFrame z kolumnami 'playerId', 'gameId', 'rating'
        """
        # Przygotuj dane
        self.users = ratings_df['playerId'].unique()
        self.items = ratings_df['gameId'].unique()
        self.user_to_idx = {u: i for i, u in enumerate(self.users)}
        self.item_to_idx = {i: j for j, i in enumerate(self.items)}
        
        n_users = len(self.users)
        n_items = len(self.items)
        
        # Inicjalizacja
        np.random.seed(42)
        self.user_factors = np.random.normal(0, 0.1, (n_users, self.n_factors))
        self.item_factors = np.random.normal(0, 0.1, (n_items, self.n_factors))
        self.user_bias = np.zeros(n_users)
        self.item_bias = np.zeros(n_items)
        self.global_mean = ratings_df['rating'].mean()
        
        # Trenowanie z użyciem SGD
        ratings = ratings_df.values
        n_ratings = len(ratings)
        
        for epoch in range(self.n_epochs):
            total_error = 0
            # Losowa kolejność
            indices = np.random.permutation(n_ratings)
            
            for idx in indices:
                user_id, item_id, rating = ratings[idx]
                u = self.user_to_idx[user_id]
                i = self.item_to_idx[item_id]
                
                # Przewidywanie
                pred = self.global_mean + self.user_bias[u] + self.item_bias[i]
                pred += np.dot(self.user_factors[u], self.item_factors[i])
                
                # Błąd
                error = rating - pred
                total_error += error ** 2
                
                # Aktualizacja gradient descent
                self.user_bias[u] += self.learning_rate * (error - self.regularization * self.user_bias[u])
                self.item_bias[i] += self.learning_rate * (error - self.regularization * self.item_bias[i])
                
                # Aktualizacja czynników
                user_factor = self.user_factors[u]
                item_factor = self.item_factors[i]
                
                self.user_factors[u] += self.learning_rate * (error * item_factor - self.regularization * user_factor)
                self.item_factors[i] += self.learning_rate * (error * user_factor - self.regularization * item_factor)
            
            # Zmniejsz learning rate
            self.learning_rate *= 0.95
            
            # Wyświetl postęp
            if (epoch + 1) % 10 == 0:
                rmse = np.sqrt(total_error / n_ratings)
                print(f"Epoch {epoch + 1}/{self.n_epochs}, RMSE: {rmse:.4f}")
    
    def predict(self, user_id, item_id):
        """Przewiduj ocenę dla pary (użytkownik, gra)"""
        if user_id not in self.user_to_idx or item_id not in self.item_to_idx:
            return self.global_mean
        
        u = self.user_to_idx[user_id]
        i = self.item_to_idx[item_id]
        
        pred = self.global_mean + self.user_bias[u] + self.item_bias[i]
        pred += np.dot(self.user_factors[u], self.item_factors[i])
        
        # Ograniczenie do skali 0-10
        return max(0, min(10, pred))
    
    def get_recommendations(self, user_id, n_recommendations=5):
        """Pobierz rekomendacje dla użytkownika"""
        if user_id not in self.user_to_idx:
            return f"Użytkownik {user_id} nie istnieje"
        
        # Znajdź gry już ocenione przez użytkownika
        user_games = df[df['playerId'] == user_id]['gameId'].values
        
        # Wszystkie gry nieocenione przez użytkownika
        all_games = self.items
        unrated_games = [g for g in all_games if g not in user_games]
        
        # Przewiduj oceny
        predictions = []
        for game_id in unrated_games:
            pred = self.predict(user_id, game_id)
            predictions.append((game_id, pred))
        
        # Sortuj i zwróć rekomendacje
        recommendations = sorted(predictions, key=lambda x: x[1], reverse=True)[:n_recommendations]
        return recommendations

def svd_cf(user_id, n_recommendations=5, n_factors=20, n_epochs=30):
    """
    Matrix Factorization (SVD) - własna implementacja
    """
    # Przygotuj dane
    ratings_data = df[['playerId', 'gameId', 'rating']].copy()
    
    # Trenuj model
    model = CustomSVD(n_factors=n_factors, n_epochs=n_epochs)
    print("Trenowanie modelu SVD...")
    model.fit(ratings_data)
    
    # Pobierz rekomendacje
    recommendations = model.get_recommendations(user_id, n_recommendations)
    return recommendations

### Testowanie

In [31]:
def test_all_methods(user_id=0, n_recommendations=5):
    """
    Testuj wszystkie metody i wyświetl wyniki
    """
    print(f"\n{'='*70}")
    print(f"🎮 REKOMENDACJE DLA UŻYTKOWNIKA {user_id}")
    print(f"{'='*70}\n")
    
    # 1. User-Based z różnymi metrykami
    print("📌 USER-BASED COLLABORATIVE FILTERING")
    print("-" * 50)
    for metric in ['cosine', 'pearson', 'euclidean']:
        print(f"\n   Metryka: {metric.upper()}")
        recommendations = user_based_cf(user_id, n_recommendations, similarity=metric)
        for i, (game_id, score) in enumerate(recommendations, 1):
            print(f"   {i}. Gra {game_id} - przewidywana ocena: {score:.2f}")
    
    # 2. Item-Based z różnymi metrykami
    print("\n📌 ITEM-BASED COLLABORATIVE FILTERING")
    print("-" * 50)
    for metric in ['cosine', 'pearson', 'euclidean']:
        print(f"\n   Metryka: {metric.upper()}")
        recommendations = item_based_cf(user_id, n_recommendations, similarity=metric)
        for i, (game_id, score) in enumerate(recommendations, 1):
            print(f"   {i}. Gra {game_id} - przewidywana ocena: {score:.2f}")
    
    # 3. SVD
    print("\n📌 MATRIX FACTORIZATION (SVD)")
    print("-" * 50)
    recommendations = svd_cf(user_id, n_recommendations, n_factors=15, n_epochs=20)
    for i, (game_id, score) in enumerate(recommendations, 1):
        print(f"   {i}. Gra {game_id} - przewidywana ocena: {score:.2f}")
    
    print("\n" + "="*70)

# Uruchom test dla użytkownika 0
test_all_methods(user_id=0, n_recommendations=5)


🎮 REKOMENDACJE DLA UŻYTKOWNIKA 0

📌 USER-BASED COLLABORATIVE FILTERING
--------------------------------------------------

   Metryka: COSINE
   1. Gra 16 - przewidywana ocena: 7.52
   2. Gra 2 - przewidywana ocena: 7.43
   3. Gra 29 - przewidywana ocena: 6.76
   4. Gra 35 - przewidywana ocena: 6.72
   5. Gra 44 - przewidywana ocena: 4.80

   Metryka: PEARSON
   1. Gra 2 - przewidywana ocena: 7.42
   2. Gra 16 - przewidywana ocena: 7.37
   3. Gra 29 - przewidywana ocena: 6.89
   4. Gra 44 - przewidywana ocena: 6.67
   5. Gra 35 - przewidywana ocena: 6.47

   Metryka: EUCLIDEAN
   1. Gra 6 - przewidywana ocena: 8.23
   2. Gra 18 - przewidywana ocena: 8.11
   3. Gra 25 - przewidywana ocena: 8.10
   4. Gra 5 - przewidywana ocena: 8.10
   5. Gra 10 - przewidywana ocena: 7.88

📌 ITEM-BASED COLLABORATIVE FILTERING
--------------------------------------------------

   Metryka: COSINE
   1. Gra 16 - przewidywana ocena: 7.00
   2. Gra 29 - przewidywana ocena: 7.00
   3. Gra 35 - przewidywana 

In [35]:
def get_recommendations(user_id, n=5, method='user', similarity='cosine'):
    """
    Prosta funkcja do szybkiego uzyskania rekomendacji
    
    Parameters:
    - user_id: ID użytkownika
    - n: liczba rekomendacji
    - method: 'user', 'item', 'svd', 'hybrid'
    - similarity: 'cosine', 'pearson', 'euclidean'
    """
    if method == 'user':
        return user_based_cf(user_id, n, similarity=similarity)
    elif method == 'item':
        return item_based_cf(user_id, n, similarity=similarity)
    elif method == 'svd':
        return svd_cf(user_id, n)
    else:
        return f"Nieznana metoda: {method}"

# Przykład użycia
print("\n🎮 SZYBKIE REKOMENDACJE")
print("=" * 40)
user_id = 5
recommendations = get_recommendations(user_id, n=5, method='user')
print(f"\nRekomendacje dla użytkownika {user_id}:")
for i, (game_id, score) in enumerate(recommendations, 1):
    print(f"{i}. Gra {game_id} - ocena: {score:.2f}")


🎮 SZYBKIE REKOMENDACJE

Rekomendacje dla użytkownika 5:
1. Gra 36 - ocena: 8.65
2. Gra 14 - ocena: 7.93
3. Gra 25 - ocena: 7.17
4. Gra 3 - ocena: 6.28
